# Advanced Flood Streamflow Prediction - Ensemble Model
## Ganga-Brahmaputra Basin Challenge

**Goal:** Achieve KGE > 0.9996 by using ensemble of XGBoost, LightGBM, and Neural Network

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
import lightgbm as lgb
from sklearn.preprocessing import RobustScaler

import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, LSTM, Input, concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import matplotlib.pyplot as plt

np.random.seed(42)
tf.random.set_seed(42)

print('All imports successful')

In [ ]:
print('Loading data...')
train_df = pd.read_csv('train_flood.csv')
test_df = pd.read_csv('test_flood.csv')

print(f'Train shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')

In [ ]:
def kge(obs, sim):
    r = np.corrcoef(obs, sim)[0, 1]
    alpha = np.std(sim) / np.std(obs)
    beta = np.mean(sim) / np.mean(obs)
    kge_val = 1 - np.sqrt(r**2 + (alpha - 1)**2 + (beta - 1)**2)
    return kge_val, r, alpha, beta

def nse(obs, sim):
    ss_res = np.sum((obs - sim) ** 2)
    ss_tot = np.sum((obs - np.mean(obs)) ** 2)
    return 1 - (ss_res / ss_tot)

print('Metrics defined')

In [ ]:
def engineer_features(df, target_col='streamflow_tomorrow_cumecs'):
    df_feat = df.copy()
    has_target = target_col in df_feat.columns
    if has_target:
        target = df_feat[target_col]
        df_feat = df_feat.drop(columns=[target_col])
    
    rainfall_cols = [col for col in df_feat.columns if 'rainfall' in col.lower() or 'precip' in col.lower()]
    flow_cols = [col for col in df_feat.columns if 'flow' in col.lower() or 'cumec' in col.lower()]
    saturation_cols = [col for col in df_feat.columns if 'satur' in col.lower()]
    
    df_feat = df_feat.fillna(method='ffill').fillna(method='bfill').fillna(0)
    
    if rainfall_cols and flow_cols:
        for rain_col in rainfall_cols:
            for flow_col in flow_cols:
                df_feat[f'{rain_col}_x_{flow_col}'] = df_feat[rain_col] * df_feat[flow_col]
                df_feat[f'{rain_col}_div_{flow_col}'] = df_feat[rain_col] / (df_feat[flow_col] + 1)
    
    if flow_cols:
        for flow_col in flow_cols:
            df_feat[f'{flow_col}_rate_change'] = df_feat[flow_col].diff().fillna(0)
            df_feat[f'{flow_col}_pct_change'] = df_feat[flow_col].pct_change().fillna(0)
            df_feat[f'{flow_col}_rolling_3d'] = df_feat[flow_col].rolling(window=3, min_periods=1).mean()
            df_feat[f'{flow_col}_rolling_7d'] = df_feat[flow_col].rolling(window=7, min_periods=1).mean()
    
    if rainfall_cols:
        for rain_col in rainfall_cols:
            for window in [3, 7, 15, 30]:
                df_feat[f'{rain_col}_cum_{window}d'] = df_feat[rain_col].rolling(window=window, min_periods=1).sum()
            api = pd.Series(0, index=df_feat.index)
            for i in range(1, len(df_feat)):
                api.iloc[i] = 0.8 * api.iloc[i-1] + df_feat[rain_col].iloc[i]
            df_feat[f'{rain_col}_api'] = api
    
    df_feat = df_feat.replace([np.inf, -np.inf], 0)
    
    if has_target:
        df_feat[target_col] = target
    
    return df_feat

print('Feature engineering function defined')

In [ ]:
print('Applying feature engineering...')
train_feat = engineer_features(train_df, target_col='streamflow_tomorrow_cumecs')
test_feat = engineer_features(test_df)

common_cols = list(set(train_feat.columns) & set(test_feat.columns))
train_feat = train_feat[common_cols + ['streamflow_tomorrow_cumecs']]
test_feat = test_feat[common_cols]

print(f'Final train shape: {train_feat.shape}')
print(f'Final test shape: {test_feat.shape}')

In [ ]:
split_idx = int(0.8 * len(train_feat))

X_train = train_feat.iloc[:split_idx, :-1]
y_train = train_feat.iloc[:split_idx, -1]

X_val = train_feat.iloc[split_idx:, :-1]
y_val = train_feat.iloc[split_idx:, -1]

X_test = test_feat.copy()

scaler_X = RobustScaler()
scaler_y = RobustScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)
X_test_scaled = scaler_X.transform(X_test)

y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_val_scaled = scaler_y.transform(y_val.values.reshape(-1, 1)).ravel()

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

In [ ]:
print('Training XGBoost...')
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train_scaled, y_train_scaled,
    eval_set=[(X_val_scaled, y_val_scaled)],
    early_stopping_rounds=50,
    verbose=False
)

y_val_pred_xgb_scaled = xgb_model.predict(X_val_scaled)
y_val_pred_xgb = scaler_y.inverse_transform(y_val_pred_xgb_scaled.reshape(-1, 1)).ravel()
kge_xgb, r_xgb, alpha_xgb, beta_xgb = kge(y_val.values, y_val_pred_xgb)
nse_xgb = nse(y_val.values, y_val_pred_xgb)

print(f'XGBoost - KGE: {kge_xgb:.6f}, NSE: {nse_xgb:.6f}')

In [ ]:
print('Training LightGBM...')
lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.05,
    num_leaves=50,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

lgb_model.fit(
    X_train_scaled, y_train_scaled,
    eval_set=[(X_val_scaled, y_val_scaled)],
    early_stopping_rounds=50,
    verbose_eval=False
)

y_val_pred_lgb_scaled = lgb_model.predict(X_val_scaled)
y_val_pred_lgb = scaler_y.inverse_transform(y_val_pred_lgb_scaled.reshape(-1, 1)).ravel()
kge_lgb, r_lgb, alpha_lgb, beta_lgb = kge(y_val.values, y_val_pred_lgb)
nse_lgb = nse(y_val.values, y_val_pred_lgb)

print(f'LightGBM - KGE: {kge_lgb:.6f}, NSE: {nse_lgb:.6f}')

In [ ]:
print('Building Neural Network...')

def build_nn_model(input_dim):
    dense_input = Input(shape=(input_dim,))
    dense_x = Dense(128, activation='relu')(dense_input)
    dense_x = Dropout(0.3)(dense_x)
    dense_x = Dense(64, activation='relu')(dense_x)
    dense_x = Dropout(0.3)(dense_x)
    dense_x = Dense(32, activation='relu')(dense_x)
    
    lstm_input = Input(shape=(input_dim,))
    lstm_x = tf.keras.layers.Reshape((input_dim, 1))(lstm_input)
    lstm_x = LSTM(64, return_sequences=True)(lstm_x)
    lstm_x = Dropout(0.3)(lstm_x)
    lstm_x = LSTM(32)(lstm_x)
    lstm_x = Dropout(0.3)(lstm_x)
    
    merged = concatenate([dense_x, lstm_x])
    merged = Dense(32, activation='relu')(merged)
    merged = Dropout(0.2)(merged)
    output = Dense(1, activation='linear')(merged)
    
    model = Model(inputs=[dense_input, lstm_input], outputs=output)
    return model

nn_model = build_nn_model(X_train_scaled.shape[1])
nn_model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
print('Model built')

In [ ]:
print('Training Neural Network...')

early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=0.00001)

history = nn_model.fit(
    [X_train_scaled, X_train_scaled],
    y_train_scaled,
    validation_data=([X_val_scaled, X_val_scaled], y_val_scaled),
    epochs=200,
    batch_size=256,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

y_val_pred_nn_scaled = nn_model.predict([X_val_scaled, X_val_scaled], verbose=0).ravel()
y_val_pred_nn = scaler_y.inverse_transform(y_val_pred_nn_scaled.reshape(-1, 1)).ravel()

kge_nn, r_nn, alpha_nn, beta_nn = kge(y_val.values, y_val_pred_nn)
nse_nn = nse(y_val.values, y_val_pred_nn)

print(f'Neural Network - KGE: {kge_nn:.6f}, NSE: {nse_nn:.6f}')

In [ ]:
from scipy.optimize import minimize

def kge_negative(weights, pred_xgb, pred_lgb, pred_nn, obs):
    w = np.abs(weights) / np.sum(np.abs(weights))
    ensemble_pred = w[0] * pred_xgb + w[1] * pred_lgb + w[2] * pred_nn
    kge_val, _, _, _ = kge(obs, ensemble_pred)
    return -kge_val

print('Optimizing ensemble weights...')
init_weights = np.array([0.33, 0.33, 0.34])
result = minimize(
    kge_negative,
    init_weights,
    args=(y_val_pred_xgb, y_val_pred_lgb, y_val_pred_nn, y_val.values),
    method='Nelder-Mead'
)

optimal_weights = np.abs(result.x) / np.sum(np.abs(result.x))
print(f'Optimal weights: XGB={optimal_weights[0]:.4f}, LGB={optimal_weights[1]:.4f}, NN={optimal_weights[2]:.4f}')

y_val_pred_ensemble = (optimal_weights[0] * y_val_pred_xgb + optimal_weights[1] * y_val_pred_lgb + optimal_weights[2] * y_val_pred_nn)
kge_ens, r_ens, alpha_ens, beta_ens = kge(y_val.values, y_val_pred_ensemble)
nse_ens = nse(y_val.values, y_val_pred_ensemble)

print(f'ENSEMBLE - KGE: {kge_ens:.6f}, NSE: {nse_ens:.6f}')

In [ ]:
print('Regime-specific analysis...')
q25 = y_val.quantile(0.25)
q75 = y_val.quantile(0.75)
q95 = y_val.quantile(0.95)

regimes = {
    'Low Flow': y_val.values < q25,
    'High Flow': (y_val.values >= q75) & (y_val.values < q95),
    'Flood Peak': y_val.values >= q95
}

for regime_name, mask in regimes.items():
    if mask.sum() > 0:
        obs_regime = y_val.values[mask]
        pred_regime = y_val_pred_ensemble[mask]
        kge_regime, _, _, _ = kge(obs_regime, pred_regime)
        print(f'{regime_name}: KGE={kge_regime:.5f}')

In [ ]:
print('Generating final predictions...')

X_full = pd.concat([X_train, X_val])
y_full = pd.concat([y_train, y_val])
X_full_scaled = scaler_X.transform(X_full)
y_full_scaled = scaler_y.transform(y_full.values.reshape(-1, 1)).ravel()

xgb_final = xgb.XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_alpha=1.0, reg_lambda=1.0, random_state=42, n_jobs=-1)
xgb_final.fit(X_full_scaled, y_full_scaled, verbose=False)
y_test_pred_xgb_scaled = xgb_final.predict(X_test_scaled)
y_test_pred_xgb = scaler_y.inverse_transform(y_test_pred_xgb_scaled.reshape(-1, 1)).ravel()

lgb_final = lgb.LGBMRegressor(n_estimators=500, max_depth=7, learning_rate=0.05, num_leaves=50, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)
lgb_final.fit(X_full_scaled, y_full_scaled, verbose_eval=False)
y_test_pred_lgb_scaled = lgb_final.predict(X_test_scaled)
y_test_pred_lgb = scaler_y.inverse_transform(y_test_pred_lgb_scaled.reshape(-1, 1)).ravel()

nn_final = build_nn_model(X_full_scaled.shape[1])
nn_final.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
nn_final.fit([X_full_scaled, X_full_scaled], y_full_scaled, epochs=100, batch_size=256, verbose=0)
y_test_pred_nn_scaled = nn_final.predict([X_test_scaled, X_test_scaled], verbose=0).ravel()
y_test_pred_nn = scaler_y.inverse_transform(y_test_pred_nn_scaled.reshape(-1, 1)).ravel()

y_test_pred_ensemble = (optimal_weights[0] * y_test_pred_xgb + optimal_weights[1] * y_test_pred_lgb + optimal_weights[2] * y_test_pred_nn)

print(f'Test predictions - Min: {y_test_pred_ensemble.min():.2f}, Max: {y_test_pred_ensemble.max():.2f}')
y_test_pred_ensemble = np.clip(y_test_pred_ensemble, 0.1, None)

In [ ]:
print('Creating submission...')
row_ids = test_df['row_id'].values if 'row_id' in test_df.columns else np.arange(len(test_df))

submission = pd.DataFrame({
    'row_id': row_ids,
    'streamflow_tomorrow_cumecs': y_test_pred_ensemble
})

submission.to_csv('submission.csv', index=False)
print(f'Submission saved: submission.csv')
print(submission.head())

In [ ]:
print('\n=== SUMMARY ===')
print(f'Ensemble KGE: {kge_ens:.6f}')
print(f'Ensemble NSE: {nse_ens:.6f}')
print(f'\nWeights: XGB={optimal_weights[0]:.4f}, LGB={optimal_weights[1]:.4f}, NN={optimal_weights[2]:.4f}')
print(f'\nSubmission rows: {len(submission)}')
print('\nReady to submit to leaderboard!')